#### Analyze the `NicheCompass` results on Xenium adult healthy colon data add-on dataset (425 probes)
- **Developed by:** Anna Maguza
- **Affilation:** Faculty of Medicine, Würzburg University
- **Created date:** 25th March 2024
- **Last modified dte:** 26th March 2025

This notebook is created to analyze niche composition and gene programs in the stem cells niche.

#### Import Libraries

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import random
import warnings
from datetime import datetime

import matplotlib.pyplot as plt
from pywaffle import Waffle
import pandas as pd
import scanpy as sc
import seaborn as sns
import squidpy as sq
from matplotlib import gridspec
from sklearn.preprocessing import MinMaxScaler
import numpy as np
import anndata as ad

from nichecompass.models import NicheCompass
from nichecompass.utils import (add_gps_from_gp_dict_to_adata,
                                compute_communication_gp_network,
                                visualize_communication_gp_network,
                                create_new_color_dict,
                                extract_gp_dict_from_mebocost_ms_interactions,
                                extract_gp_dict_from_nichenet_lrt_interactions,
                                extract_gp_dict_from_omnipath_lr_interactions,
                                filter_and_combine_gp_dict_gps_v2,
                                generate_enriched_gp_info_plots)

#### Define Parameters

In [4]:
latent_key = "nichecompass_latent"

gp_names_key = "nichecompass_gp_names"
cell_type_key = "C_scANVI"
latent_leiden_resolution = 0.4
latent_cluster_key = f"latent_leiden_{str(latent_leiden_resolution)}"
sample_key = "Donor_ID"
spot_size = 0.2
differential_gp_test_results_key = "nichecompass_differential_gp_test_results"

#### Run Notebook Setup

In [5]:
warnings.filterwarnings("ignore")

#### Configure Paths

In [3]:
load_timestamp = '26032025_173236'

In [ ]:
artifacts_folder_path = f"NicheCompass/artifacts"
model_folder_path = f"{artifacts_folder_path}/single_sample/{load_timestamp}/model"
figure_folder_path = f"{artifacts_folder_path}/single_sample/{load_timestamp}/figures"

#### Analysis

In [22]:
model = NicheCompass.load(dir_path=model_folder_path,
                          adata=None,
                          adata_file_name="adata.h5ad",
                          gp_names_key=gp_names_key)

--- INITIALIZING NEW NETWORK MODULE: VARIATIONAL GENE PROGRAM GRAPH AUTOENCODER ---
LOSS -> include_edge_recon_loss: True, include_gene_expr_recon_loss: True, rna_recon_loss: nb
NODE LABEL METHOD -> one-hop-norm
ACTIVE GP THRESHOLD RATIO -> 0.01
LOG VARIATIONAL -> True
ONE HOP GCN NORM RNA NODE LABEL AGGREGATOR
ENCODER -> n_input: 425, n_cat_covariates_embed_input: 0, n_hidden: 211, n_latent: 111, n_addon_latent: 100, n_fc_layers: 1, n_layers: 1, conv_layer: gcnconv, n_attention_heads: 0, dropout_rate: 0.0, 
COSINE SIM GRAPH DECODER -> dropout_rate: 0.0
MASKED TARGET RNA DECODER -> n_prior_gp_input: 111, n_addon_gp_input: 100, n_cat_covariates_embed_input: 0, n_output: 425
MASKED SOURCE RNA DECODER -> n_prior_gp_input: 111, n_addon_gp_input: 100, n_cat_covariates_embed_input: 0, n_output: 425


In [23]:
samples = model.adata.obs[sample_key].unique().tolist()

In [24]:
model.adata

AnnData object with n_obs × n_vars = 274037 × 425
    obs: 'Study_name', 'Donor_ID', 'Library_Preparation_Protocol', 'dataset', '_scvi_batch', '_scvi_labels', 'seed_labels', 'C_scANVI', 'SC_subsets', 'Cell_State', 'cell_id', 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'n_counts', 'REG4_score', 'gdT', 'Endothelial cells', 'latent_leiden_0.4'
    var: 'gene_ids', 'feature_types', 'genome'
    uns: 'B cells_colors', 'BEST4+ epithelial_colors', 'CD4 T_colors', 'CD8 T_colors', 'C_scANVI_colors', 'Cell_State_colors', 'Colonocyte_colors', 'DC_colors', 'Deep_crypt_secretory_colors', 'Donor_ID_colors', 'EECs_colors', 'Endothelial cells_colors', 'Enterocyte_colors', 'FXYD3+_CKB+_SC_colors', 'Fibroblasts_colors', 'Glial cells_colors', 'Goblet cells_colors', 'ILCs_colors', 'LEC_colors', 'Library_Preparation_Protocol_colors', 'MTRNR2L12+ASS

#### Visualize NicheCompass Latent GP Space

In [25]:
cell_type_colors = create_new_color_dict(
    adata=model.adata,
    cat_key=cell_type_key)

In [26]:
adata = model.adata.copy()

In [ ]:
# Create plot of cell type annotations in physical and latent space
groups = None
save_fig = True
file_path = f"{figure_folder_path}/" \
            "cell_types_latent_physical_space.png"

fig = plt.figure(figsize=(12, 14), dpi=300)
title = fig.suptitle(t=f"Cell Types " \
                       "in Latent and Physical Space",
                     y=0.96,
                     x=0.55,
                     fontsize=20)
spec1 = gridspec.GridSpec(ncols=1,
                          nrows=2,
                          width_ratios=[1],
                          height_ratios=[3, 2])
spec2 = gridspec.GridSpec(ncols=len(samples),
                          nrows=2,
                          width_ratios=[1] * len(samples),
                          height_ratios=[3, 2])
axs = []
axs.append(fig.add_subplot(spec1[0]))
sc.pl.umap(adata=model.adata, 
           color=[cell_type_key],
           groups=groups,palette=cell_type_colors,
           title=f"Cell Types in Latent Space",
           ax=axs[0],
           show=False)
for idx, sample in enumerate(samples):
    axs.append(fig.add_subplot(spec2[len(samples) + idx]))
    sq.pl.spatial_scatter(adata, 
              library_id="spatial", 
              shape=None,
              color="C_scANVI",
              size=0.2,
              title=f"Cell Types in Physical Space \n"
                    f"(Sample: {sample})",
              legend_loc=None, 
              frameon=False,
              ax=axs[idx+1], 
              alpha=1.0)

# Create and position shared legend
handles, labels = axs[0].get_legend_handles_labels()
lgd = fig.legend(handles,
                 labels,
                 loc="center left",
                 bbox_to_anchor=(0.98, 0.5))
axs[0].get_legend().remove()

# Adjust, save and display plot
plt.subplots_adjust(wspace=0.2, hspace=0.25)
if save_fig:
    fig.savefig(file_path,
                bbox_extra_artists=(lgd, title),
                bbox_inches="tight")
plt.show()

#### 4.2 Identify Niches

+ Compute latent Leiden clustering

In [31]:
sc.tl.leiden(adata=model.adata,
             resolution=latent_leiden_resolution,
             key_added=latent_cluster_key,
             neighbors_key=latent_key)

In [32]:
latent_cluster_colors = create_new_color_dict(
    adata=model.adata,
    cat_key=latent_cluster_key)

In [33]:
adata = model.adata.copy()

In [35]:
adata.uns.pop('latent_leiden_0.4_colors')

array(['#f6cf71ff', '#87c55fff', '#f89c74ff', '#ff4d4dff', '#8be0a4ff',
       '#c38d9eff', '#d3b484ff', '#dab6c4ff', '#66c5ccff', '#b3b3b3ff',
       '#b497e7ff', '#fe88b1ff', '#c9db74ff', '#dcb0f2ff', '#9eb9f3ff',
       '#276a8cff', '#9b4dcaff', '#9d88a2ff'], dtype=object)

In [ ]:
# Create plot of latent cluster / niche annotations in physical and latent space
groups = None # set this to a specific cluster for easy visualization, e.g. ["17"]
save_fig = True
file_path = f"{figure_folder_path}/" \
            f"res_{latent_leiden_resolution}_" \
            "niches_latent_physical_space.png"

fig = plt.figure(figsize=(12, 14), dpi = 300)
title = fig.suptitle(t=f"NicheCompass Niches " \
                       "in Latent and Physical Space",
                     y=0.96,
                     x=0.55,
                     fontsize=20)
spec1 = gridspec.GridSpec(ncols=1,
                          nrows=2,
                          width_ratios=[1],
                          height_ratios=[3, 2])
spec2 = gridspec.GridSpec(ncols=len(samples),
                          nrows=2,
                          width_ratios=[1] * len(samples),
                          height_ratios=[3, 2])
axs = []
axs.append(fig.add_subplot(spec1[0]))
sc.pl.umap(adata=model.adata,
           color=[latent_cluster_key],
           groups=groups,
           palette=latent_cluster_colors,
           title=f"Niches in Latent Space",
           ax=axs[0],
           show=False)
for idx, sample in enumerate(samples):
    axs.append(fig.add_subplot(spec2[len(samples) + idx]))
    sq.pl.spatial_scatter(adata,
                          library_id="spatial",
                          shape=None,
                          color=[latent_cluster_key],
                          size=0.2,
                          title=f"Niches in Physical Space \n"
                              f"(Sample: {sample})",
                          legend_loc=None,
                          frameon=False,
                          ax=axs[idx+1])

# Create and position shared legend
handles, labels = axs[0].get_legend_handles_labels()
lgd = fig.legend(handles,
                 labels,
                 loc="center left",
                 bbox_to_anchor=(0.98, 0.5))
axs[0].get_legend().remove()

# Adjust, save and display plot
plt.subplots_adjust(wspace=0.2, hspace=0.25)
if save_fig:
    fig.savefig(file_path,
                bbox_extra_artists=(lgd, title),
                bbox_inches="tight")
plt.show()

+ Get closer look on the niches

In [ ]:
plt.rcParams['figure.dpi'] = 300
plt.rcParams['figure.figsize'] = (10, 15)

sq.pl.spatial_scatter(
    adata,
    library_id="spatial",
    shape=None, 
    color="latent_leiden_0.4",
    size=0.1,
    frameon=False,
    alpha=1.0,
    #crop_coord=[(6300, 1950, 7600, 2950)], 
)
plt.savefig(f"{figure_folder_path}/predicted_niches.png", bbox_inches="tight", dpi=300)

In [ ]:
plt.rcParams['figure.dpi'] = 300
plt.rcParams['figure.figsize'] = (7, 7)

sq.pl.spatial_scatter(
    adata,
    library_id="spatial",
    shape=None, 
    color="latent_leiden_0.4",
    size=20,
    frameon=False,
    alpha=1.0,
    crop_coord=[(6300, 1950, 7600, 2950)], 
)
plt.savefig(f"NicheCompass/artifacts/single_sample/26032025_173236/predicted_niches_zoomed_lymph_node2.png", bbox_inches="tight", dpi=300)

# 4.3 Characterize Niches

### 4.3.1 Niche Composition

In [42]:
df_counts = adata.obs.groupby(['latent_leiden_0.4', 'C_scANVI']).size().unstack()

In [43]:
df_counts.head(22)

C_scANVI,B cells,BEST4+ epithelial,CD4 T,CD8 T,Colonocyte,DC,EECs,Endothelial cells,Fibroblasts,Glial cells,...,Microfold cell,Monocytes,Myofibroblasts,NK,Pericytes,Plasma cells,Stem cells,TA,Tuft cells,Arterial capillary
latent_leiden_0.4,,,,,,,,,,,,,,,,,,,,,
0,1,0,17,44,0,15,0,401,2823,1083,...,0,8,25402,5,215,3,0,0,0,0
1,109,121,312,161,1171,45,1142,706,2838,1270,...,0,5,371,22,70,856,7061,3764,1687,16
2,421,1,1037,246,16,50,1671,750,3102,1011,...,0,11,577,14,113,429,6791,972,2675,18
3,138,0,223,134,19,29,2,1223,8870,592,...,0,77,110,33,4507,103,6,9,4,8
4,59,25,1522,433,1762,357,32,992,4711,974,...,0,2,263,14,32,8339,22,127,7,233
5,0,2600,1,85,17009,9,76,1,1,0,...,1,40,0,212,0,40,0,0,1,28
6,10967,0,6358,215,12,585,0,311,830,85,...,0,3,156,3,25,561,3,2,0,13
7,26,16,1171,250,1578,2045,3,843,2023,133,...,0,3,38,58,23,4221,0,12,0,2649
8,1,726,72,111,9347,19,633,81,391,184,...,0,0,34,23,3,668,812,976,321,7


In [44]:
df_counts.to_csv(f"{figure_folder_path}/niche_composition.csv")

In [46]:
C_scANVI_colors = {
    'Fibroblasts': "#ff9cda",       # plum - fibroblasts
    'Myofibroblasts': "#FFE2E2",    # mediumpurple
    'Pericytes': "#008B8B",         # plum
    'Glial cells': "#CD853F",       # peru
    'Macrophages': "#9eb9f3",       # lightsteelblue
    'CD8 T': "#7fff00",             # blueviolet
    'Endothelial cells': "#4169e1", # royalblue
    'Mast cells': "#ff8c8c",        # lightcoral
    'Monocytes': "#EB5B00",         # hotpink
    'B cells': "#7b68ee",           # mediumslateblue
    'TA': "#66c5cc",                # mediumturquoise
    'EECs': "#32cd32",              # limegreen
    'Plasma cells': "#dab6c4",      # thistle
    'Goblet cells': "#640D5F",      # chartreuse
    'Tuft cells': "#0D92F4",        # greenyellow
    'Stem cells': "#b497e7",        # darkcyan
    'CD4 T': "#006400",             # darkgreen
    'Colonocyte': "#9d88a2",        # darkgray
    'Mature venous EC': "#87c55f",  # darkseagreen
    'Mature arterial EC': "#ff00ff",# fuchsia
    'LEC': "#4B0082",               # indigo
    'BEST4+ epithelial': "#8B0000", # darkred
    'Arterial capillary': "#8be0a4",# lightgreen
    'DC': "#ffaa80",                # lightsalmon
    'NK': "#E82561",                # purple
    'Microfold cell': "#276a8c",    # teal
    'ILCs': "#A19AD3",              # tomato
    'Immune Cycling cells': "#d3b484" # tan
}

In [47]:
def get_consistent_colors(niche_df, cat_column='C_scANVI'):
    color_list = []
    for C_scANVI in niche_df[cat_column]:
        # If cell state exists in our mapping, use that color; otherwise use a default
        if C_scANVI in C_scANVI_colors:
            color_list.append(C_scANVI_colors[C_scANVI])
        else:
            # For any cell state not in our mapping, use a default color (e.g., gray)
            color_list.append("#808080")  # gray as default
    return color_list

In [ ]:
for niche_idx in range(df_counts.shape[0]): 
    niche_data = df_counts.iloc[niche_idx, 1:]  # Skip the first column
    niche_data = niche_data[niche_data >= 50]   # Filter cells with at least 50 counts
    niche_data = niche_data.sort_values(ascending=False)  
    
    niche_df = pd.DataFrame({
        'C_scANVI': niche_data.index,
        'Number_of_Cells': niche_data.values
    })
    
    color_list = get_consistent_colors(niche_df, cat_column='C_scANVI')
    
    fig = plt.figure(
        FigureClass=Waffle,
        rows=15,
        columns=70,
        values=niche_df['Number_of_Cells'],
        labels=[f"{label} ({value})" for label, value in zip(niche_df['C_scANVI'], niche_df['Number_of_Cells'])],
        legend={'loc': 'upper left', 'bbox_to_anchor': (1, 1)},
        figsize=(12, 8),
        colors=color_list
    )
    
    plt.title(f"Niche {niche_idx} Cell Composition", fontsize=14, pad=20)
    
    plt.savefig(f"{figure_folder_path}/niche_{niche_idx}.png", 
                dpi=300, 
                bbox_inches='tight')
    plt.close(fig)  # Close the figure to free memory
    
    print(f"Saved plot for niche {niche_idx}")

Saved plot for niche 0
Saved plot for niche 1
Saved plot for niche 2
Saved plot for niche 3
Saved plot for niche 4
Saved plot for niche 5
Saved plot for niche 6
Saved plot for niche 7
Saved plot for niche 8
Saved plot for niche 9
Saved plot for niche 10
Saved plot for niche 11
Saved plot for niche 12
Saved plot for niche 13
Saved plot for niche 14
Saved plot for niche 15


### 4.3.2 Differential GPs

In [49]:
# Check number of active GPs
active_gps = model.get_active_gps()
print(f"Number of total gene programs: {len(model.adata.uns[gp_names_key])}.")
print(f"Number of active gene programs: {len(active_gps)}.")

Number of total gene programs: 211.
Number of active gene programs: 128.


In [50]:
# Display example active GPs
gp_summary_df = model.get_gp_summary()
gp_summary_df[gp_summary_df["gp_active"] == True].head()

,gp_name,all_gp_idx,gp_active,active_gp_idx,n_source_genes,n_non_zero_source_genes,n_target_genes,n_non_zero_target_genes,gp_source_genes,gp_target_genes,gp_source_genes_weights,gp_target_genes_weights,gp_source_genes_importances,gp_target_genes_importances
0,CD177_ligand_receptor_GP,0,True,0,1,1,1,1,[CD177],[PECAM1],[1.7956],[0.0373],[0.9797],[0.0203]
1,CD8A_ligand_receptor_GP,1,True,1,1,1,2,2,[CD8A],"[PRF1, CEACAM5]",[2.1215],"[1.291, -0.0254]",[0.6171],"[0.3755, 0.0074]"
3,COMPLEX:DBH_SLC18A2_ligand_receptor_GP,3,True,2,1,1,1,1,[SLC18A2],[ADRA2A],[0.0231],[0.4809],[0.0458],[0.9542]
4,COMPLEX:PNMT_SLC18A2_ligand_receptor_GP,4,True,3,1,1,1,1,[SLC18A2],[ADRA2A],[0.0276],[0.6311],[0.0419],[0.9581]
5,ICOS_ligand_receptor_GP,5,True,4,1,1,1,1,[ICOS],[CD40LG],[0.0026],[0.022],[0.1068],[0.8932]


In [51]:
model.adata.uns['nichecompass_gp_summary'] = gp_summary_df

### Differential gp testing

In [52]:
selected_cats = None
comparison_cats = "rest"
title = f"NicheCompass Strongly Enriched Niche GPs"
log_bayes_factor_thresh = 2.3
save_fig = False
file_path = f"{figure_folder_path}/" \
            f"/log_bayes_factor_{log_bayes_factor_thresh}" \
             "_niches_enriched_gps_heatmap.png"

In [53]:
enriched_gps = model.run_differential_gp_tests(
    cat_key=latent_cluster_key,
    selected_cats=selected_cats,
    comparison_cats=comparison_cats,
    log_bayes_factor_thresh=log_bayes_factor_thresh)

In [54]:
model.adata.uns[differential_gp_test_results_key]

,category,gene_program,p_h0,p_h1,log_bayes_factor
0,0,CD24_ligand_receptor_target_gene_GP,0.999751,0.000249,8.297206
1,0,CXCL14_combined_GP,0.000265,0.999735,-8.236379
2,0,ANPEP_ligand_receptor_target_gene_GP,0.000520,0.999480,-7.561879
3,9,Add-on_76_GP,0.000759,0.999241,-7.183150
4,14,IL1B_combined_GP,0.998828,0.001172,6.748098
...,...,...,...,...,...
184,5,LGALS9_combined_GP,0.911768,0.088232,2.335418
185,0,UCN3_combined_GP,0.088412,0.911588,-2.333184
186,5,APOB_combined_GP,0.088891,0.911109,-2.327252
187,6,SELL_ligand_receptor_target_gene_GP,0.909380,0.090620,2.306088


In [55]:
adata = model.adata.copy()
adata

AnnData object with n_obs × n_vars = 274037 × 425
    obs: 'Study_name', 'Donor_ID', 'Library_Preparation_Protocol', 'dataset', '_scvi_batch', '_scvi_labels', 'seed_labels', 'C_scANVI', 'SC_subsets', 'Cell_State', 'cell_id', 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'n_counts', 'REG4_score', 'gdT', 'Endothelial cells', 'latent_leiden_0.4', 'CD24_ligand_receptor_target_gene_GP', 'CXCL14_combined_GP', 'ANPEP_ligand_receptor_target_gene_GP', 'Add-on_76_GP', 'IL1B_combined_GP', 'CDH1_ligand_receptor_target_gene_GP', 'Add-on_37_GP', 'Add-on_28_GP', 'SLPI_ligand_receptor_target_gene_GP', 'Add-on_77_GP', 'TIMP3_ligand_receptor_target_gene_GP', 'CLU_combined_GP', 'BMP5_combined_GP', 'TNXB_ligand_receptor_target_gene_GP', 'TFF1_combined_GP', 'TNFSF13B_combined_GP', 'CCL11_combined_GP', 'Add-on_70_GP', 'gamma-Aminobutyric acid_metaboli

In [56]:
# Convert all elements in adata.uns to strings
for key in adata.uns.keys():
    adata.uns[key] = str(adata.uns[key])

In [57]:
project = 'xenium_adult_colon'
species = 'hs'
atribute = 'NicheCompass'
name = 'AM'
counts = 'raw'
adata.write_h5ad(f"{model_folder_path}/{project}_{species}_{atribute}_{name}_{load_timestamp}_{counts}.h5ad")

### Analyze stem cells niche 1

In [79]:
# Set parameters for differential gp testing
selected_cats = ["1"]
comparison_cats = "rest"
title = f"NicheCompass Strongly Enriched Niche GPs"
log_bayes_factor_thresh = 2.3
save_fig = False
file_path = f"{figure_folder_path}/" \
            f"/log_bayes_factor_{log_bayes_factor_thresh}" \
             "_niches_enriched_gps_heatmap.svg"

In [80]:
# Run differential gp testing
enriched_gps = model.run_differential_gp_tests(
    cat_key=latent_cluster_key,
    selected_cats=selected_cats,
    comparison_cats=comparison_cats,
    log_bayes_factor_thresh=log_bayes_factor_thresh)

In [81]:
# Results are stored in a df in the adata object
model.adata.uns[differential_gp_test_results_key]

,category,gene_program,p_h0,p_h1,log_bayes_factor
0,1,ICOS_ligand_receptor_GP,0.040518,0.959482,-3.164635
1,1,Add-on_26_GP,0.952852,0.047148,3.006165
2,1,Add-on_77_GP,0.921840,0.078160,2.467609


In [ ]:
# Visualize GP activities of enriched GPs across niches
df = model.adata.obs[[latent_cluster_key] + enriched_gps].groupby(latent_cluster_key).mean()

scaler = MinMaxScaler()
normalized_columns = scaler.fit_transform(df)
normalized_df = pd.DataFrame(normalized_columns, columns=df.columns)
normalized_df.index = df.index

plt.figure(figsize=(16, 8))  # Set the figure size
ax = sns.heatmap(normalized_df,
            cmap='RdPu',
            annot=False,
            linewidths=0)
plt.xticks(rotation=45,
           fontsize=8,
           ha="right"
          )
plt.xlabel("Gene Programs", fontsize=16)
plt.savefig(f"{figure_folder_path}/enriched_gps_heatmap_niche1.png",
            bbox_inches="tight", dpi=300)

In [83]:
# Store gene program summary of enriched gene programs
save_file = True
file_path = f"{figure_folder_path}/" \
            f"/log_bayes_factor_{log_bayes_factor_thresh}_" \
            "niche1_enriched_gps_summary.csv"

gp_summary_cols = ["gp_name",
                   "n_source_genes",
                   "n_non_zero_source_genes",
                   "n_target_genes",
                   "n_non_zero_target_genes",
                   "gp_source_genes",
                   "gp_target_genes",
                   "gp_source_genes_importances",
                   "gp_target_genes_importances"]

enriched_gp_summary_df = gp_summary_df[gp_summary_df["gp_name"].isin(enriched_gps)]
cat_dtype = pd.CategoricalDtype(categories=enriched_gps, ordered=True)
enriched_gp_summary_df.loc[:, "gp_name"] = enriched_gp_summary_df["gp_name"].astype(cat_dtype)
enriched_gp_summary_df = enriched_gp_summary_df.sort_values(by="gp_name")
enriched_gp_summary_df = enriched_gp_summary_df[gp_summary_cols]

if save_file:
    enriched_gp_summary_df.to_csv(f"{file_path}")
else:
    display(enriched_gp_summary_df)

In [ ]:
plt.rcParams['figure.dpi'] = 300
plt.rcParams['figure.figsize'] = (10, 15)

sq.pl.spatial_scatter(
    adata,
    library_id="spatial",
    shape=None, 
    color="ICOS_ligand_receptor_GP",
    cmap='magma_r',
    size=0.1,
    frameon=False,
    alpha=1.0,
    #crop_coord=[(6300, 1950, 7600, 2950)], 
)
plt.savefig(f"{figure_folder_path}/ICOS_ligand_receptor_GP_niche_1.png", bbox_inches="tight", dpi=300)

In [ ]:
plt.rcParams['figure.dpi'] = 300
plt.rcParams['figure.figsize'] = (7, 7)

sq.pl.spatial_scatter(
    adata,
    library_id="spatial",
    shape=None, 
    color="ICOS_ligand_receptor_GP",
    cmap='magma_r',
    size=20,
    frameon=False,
    alpha=1.0,
    crop_coord=[(6300, 1950, 7600, 2950)], 
)
plt.savefig(f"{figure_folder_path}/ICOS_ligand_receptor_GP_niche_1_zoomed.png", bbox_inches="tight", dpi=300)

In [ ]:
plt.rcParams['figure.dpi'] = 300
plt.rcParams['figure.figsize'] = (10, 15)

sq.pl.spatial_scatter(
    adata,
    library_id="spatial",
    shape=None, 
    color="Add-on_77_GP",
    cmap='magma_r',
    size=0.1,
    frameon=False,
    alpha=1.0,
    #crop_coord=[(6300, 1950, 7600, 2950)], 
)
plt.savefig(f"{figure_folder_path}/Add-on_77_GP_niche_1.png", bbox_inches="tight", dpi=300)

In [ ]:
plt.rcParams['figure.dpi'] = 300
plt.rcParams['figure.figsize'] = (7, 7)

sq.pl.spatial_scatter(
    adata,
    library_id="spatial",
    shape=None, 
    color="Add-on_77_GP",
    cmap='magma_r',
    size=20,
    frameon=False,
    alpha=1.0,
    crop_coord=[(6300, 1950, 7600, 2950)], 
)
plt.savefig(f"{figure_folder_path}/Add-on_77_GP_niche_1_zoomed.png", bbox_inches="tight", dpi=300)

### Analyze stem cells niche 2

In [84]:
# Set parameters for differential gp testing
selected_cats = ["2"]
comparison_cats = "rest"
title = f"NicheCompass Strongly Enriched Niche GPs"
log_bayes_factor_thresh = 2.3
save_fig = False
file_path = f"{figure_folder_path}/" \
            f"/log_bayes_factor_{log_bayes_factor_thresh}" \
             "_niches_enriched_gps_heatmap.svg"

In [85]:
# Run differential gp testing
enriched_gps = model.run_differential_gp_tests(
    cat_key=latent_cluster_key,
    selected_cats=selected_cats,
    comparison_cats=comparison_cats,
    log_bayes_factor_thresh=log_bayes_factor_thresh)

In [86]:
# Results are stored in a df in the adata object
model.adata.uns[differential_gp_test_results_key]

,category,gene_program,p_h0,p_h1,log_bayes_factor
0,2,Add-on_18_GP,0.010241,0.989759,-4.571037
1,2,Add-on_26_GP,0.920817,0.079183,2.453493
2,2,FCER2_ligand_receptor_target_gene_GP,0.081412,0.918588,-2.423320
3,2,AZGP1_combined_GP,0.084796,0.915204,-2.378902
4,2,LEFTY1_combined_GP,0.912685,0.087315,2.346863


In [ ]:
# Visualize GP activities of enriched GPs across niches
df = model.adata.obs[[latent_cluster_key] + enriched_gps].groupby(latent_cluster_key).mean()

scaler = MinMaxScaler()
normalized_columns = scaler.fit_transform(df)
normalized_df = pd.DataFrame(normalized_columns, columns=df.columns)
normalized_df.index = df.index

plt.figure(figsize=(16, 8))  # Set the figure size
ax = sns.heatmap(normalized_df,
            cmap='RdPu',
            annot=False,
            linewidths=0)
plt.xticks(rotation=45,
           fontsize=8,
           ha="right"
          )
plt.xlabel("Gene Programs", fontsize=16)
plt.savefig(f"{figure_folder_path}/enriched_gps_heatmap_niche2.png",
            bbox_inches="tight", dpi=300)

In [88]:
# Store gene program summary of enriched gene programs
save_file = True
file_path = f"{figure_folder_path}/" \
            f"/log_bayes_factor_{log_bayes_factor_thresh}_" \
            "niche2_enriched_gps_summary.csv"

gp_summary_cols = ["gp_name",
                   "n_source_genes",
                   "n_non_zero_source_genes",
                   "n_target_genes",
                   "n_non_zero_target_genes",
                   "gp_source_genes",
                   "gp_target_genes",
                   "gp_source_genes_importances",
                   "gp_target_genes_importances"]

enriched_gp_summary_df = gp_summary_df[gp_summary_df["gp_name"].isin(enriched_gps)]
cat_dtype = pd.CategoricalDtype(categories=enriched_gps, ordered=True)
enriched_gp_summary_df.loc[:, "gp_name"] = enriched_gp_summary_df["gp_name"].astype(cat_dtype)
enriched_gp_summary_df = enriched_gp_summary_df.sort_values(by="gp_name")
enriched_gp_summary_df = enriched_gp_summary_df[gp_summary_cols]

if save_file:
    enriched_gp_summary_df.to_csv(f"{file_path}")
else:
    display(enriched_gp_summary_df)

In [ ]:
plt.rcParams['figure.dpi'] = 300
plt.rcParams['figure.figsize'] = (10, 15)

sq.pl.spatial_scatter(
    adata,
    library_id="spatial",
    shape=None, 
    color="LEFTY1_combined_GP",
    cmap='magma_r',
    size=0.1,
    frameon=False,
    alpha=1.0,
    #crop_coord=[(6300, 1950, 7600, 2950)], 
)
plt.savefig(f"{figure_folder_path}/LEFTY1_combined_GP_niche_2.png", bbox_inches="tight", dpi=300)

In [ ]:
plt.rcParams['figure.dpi'] = 300
plt.rcParams['figure.figsize'] = (7, 7)

sq.pl.spatial_scatter(
    adata,
    library_id="spatial",
    shape=None, 
    color="LEFTY1_combined_GP",
    cmap='magma_r',
    size=20,
    frameon=False,
    alpha=1.0,
    crop_coord=[(6300, 1950, 7600, 2950)], 
)
plt.savefig(f"{figure_folder_path}/LEFTY1_combined_GP_niche_2_zoomed.png", bbox_inches="tight", dpi=300)

### Analyze stem cells niche 8

In [101]:
# Set parameters for differential gp testing
selected_cats = ["8"]
comparison_cats = "rest"
title = f"NicheCompass Strongly Enriched Niche GPs"
log_bayes_factor_thresh = 2.3
save_fig = False
file_path = f"{figure_folder_path}/" \
            f"/log_bayes_factor_{log_bayes_factor_thresh}" \
             "_niches_enriched_gps_heatmap.svg"

In [102]:
# Run differential gp testing
enriched_gps = model.run_differential_gp_tests(
    cat_key=latent_cluster_key,
    selected_cats=selected_cats,
    comparison_cats=comparison_cats,
    log_bayes_factor_thresh=log_bayes_factor_thresh)

In [103]:
# Results are stored in a df in the adata object
model.adata.uns[differential_gp_test_results_key]

,category,gene_program,p_h0,p_h1,log_bayes_factor
0,8,Add-on_18_GP,0.981621,0.018379,3.978004
1,8,Add-on_98_GP,0.055260,0.944740,-2.838851
2,8,TTR_combined_GP,0.944467,0.055533,2.833637
3,8,CD40LG_combined_GP,0.929799,0.070201,2.583610


In [ ]:
# Visualize GP activities of enriched GPs across niches
df = model.adata.obs[[latent_cluster_key] + enriched_gps].groupby(latent_cluster_key).mean()

scaler = MinMaxScaler()
normalized_columns = scaler.fit_transform(df)
normalized_df = pd.DataFrame(normalized_columns, columns=df.columns)
normalized_df.index = df.index

plt.figure(figsize=(16, 8))  # Set the figure size
ax = sns.heatmap(normalized_df,
            cmap='RdPu',
            annot=False,
            linewidths=0)
plt.xticks(rotation=45,
           fontsize=8,
           ha="right"
          )
plt.xlabel("Gene Programs", fontsize=16)
plt.savefig(f"{figure_folder_path}/enriched_gps_heatmap_niche8.png",
            bbox_inches="tight", dpi=300)

In [105]:
# Store gene program summary of enriched gene programs
save_file = True
file_path = f"{figure_folder_path}/" \
            f"/log_bayes_factor_{log_bayes_factor_thresh}_" \
            "niche8_enriched_gps_summary.csv"

gp_summary_cols = ["gp_name",
                   "n_source_genes",
                   "n_non_zero_source_genes",
                   "n_target_genes",
                   "n_non_zero_target_genes",
                   "gp_source_genes",
                   "gp_target_genes",
                   "gp_source_genes_importances",
                   "gp_target_genes_importances"]

enriched_gp_summary_df = gp_summary_df[gp_summary_df["gp_name"].isin(enriched_gps)]
cat_dtype = pd.CategoricalDtype(categories=enriched_gps, ordered=True)
enriched_gp_summary_df.loc[:, "gp_name"] = enriched_gp_summary_df["gp_name"].astype(cat_dtype)
enriched_gp_summary_df = enriched_gp_summary_df.sort_values(by="gp_name")
enriched_gp_summary_df = enriched_gp_summary_df[gp_summary_cols]

if save_file:
    enriched_gp_summary_df.to_csv(f"{file_path}")
else:
    display(enriched_gp_summary_df)

In [ ]:
plt.rcParams['figure.dpi'] = 300
plt.rcParams['figure.figsize'] = (10, 15)

sq.pl.spatial_scatter(
    adata,
    library_id="spatial",
    shape=None, 
    color="Add-on_98_GP",
    cmap='magma_r',
    size=0.1,
    frameon=False,
    alpha=1.0,
    #crop_coord=[(6300, 1950, 7600, 2950)], 
)
plt.savefig(f"{figure_folder_path}/Add-on_98_GP_niche_8.png", bbox_inches="tight", dpi=300)

In [ ]:
plt.rcParams['figure.dpi'] = 300
plt.rcParams['figure.figsize'] = (7, 7)

sq.pl.spatial_scatter(
    adata,
    library_id="spatial",
    shape=None, 
    color="Add-on_98_GP",
    cmap='magma_r',
    size=20,
    frameon=False,
    alpha=1.0,
    crop_coord=[(6300, 1950, 7600, 2950)], 
)
plt.savefig(f"{figure_folder_path}/Add-on_98_GP_niche_8_zoomed.png", bbox_inches="tight", dpi=300)

#### Visualizations of GPs in Niche 14

In [112]:
adata = model.adata.copy()

In [113]:
adata_log = adata.copy()
sc.pp.normalize_total(adata_log)
sc.pp.log1p(adata_log)

In [115]:
# Set parameters for differential gp testing
selected_cats = None
comparison_cats = "rest"
title = f"NicheCompass Strongly Enriched Niche GPs"
log_bayes_factor_thresh = 2.3
save_fig = False
file_path = f"{figure_folder_path}/" \
            f"/log_bayes_factor_{log_bayes_factor_thresh}" \
             "_niches_enriched_gps_heatmap.svg"

In [117]:
# Run differential gp testing
enriched_gps = model.run_differential_gp_tests(
    cat_key=latent_cluster_key,
    selected_cats=selected_cats,
    comparison_cats=comparison_cats,
    log_bayes_factor_thresh=log_bayes_factor_thresh)

In [120]:
df = model.adata.uns[differential_gp_test_results_key]
df.to_csv(f"{figure_folder_path}/all_enriched_gps_logbayes_factors.csv")

In [119]:
save_file = True
file_path = f"{figure_folder_path}/" \
            f"/log_bayes_factor_{log_bayes_factor_thresh}_" \
            "all_enriched_gps_summary.csv"

gp_summary_cols = ["gp_name",
                   "n_source_genes",
                   "n_non_zero_source_genes",
                   "n_target_genes",
                   "n_non_zero_target_genes",
                   "gp_source_genes",
                   "gp_target_genes",
                   "gp_source_genes_importances",
                   "gp_target_genes_importances"]

enriched_gp_summary_df = gp_summary_df[gp_summary_df["gp_name"].isin(enriched_gps)]
cat_dtype = pd.CategoricalDtype(categories=enriched_gps, ordered=True)
enriched_gp_summary_df.loc[:, "gp_name"] = enriched_gp_summary_df["gp_name"].astype(cat_dtype)
enriched_gp_summary_df = enriched_gp_summary_df.sort_values(by="gp_name")
enriched_gp_summary_df = enriched_gp_summary_df[gp_summary_cols]

if save_file:
    enriched_gp_summary_df.to_csv(f"{file_path}")
else:
    display(enriched_gp_summary_df)